# Chess Puzzle Difficulty Classification
## IS-212 Data Mining Project

**Dataset:** Lichess Database Puzzles (50,000 sample)
**Goal:** Predict puzzle difficulty (Easy/Medium/Hard/Expert) + Discover association rules

---

## Google Colab Setup

In [ ]:
!pip install python-chess mlxtend

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CHANGE THIS to your file path in Google Drive
DATA_PATH = '/content/drive/MyDrive/Datamining/lichess_db_puzzle_sample.csv'

## Phase 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import chess
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, MultiLabelBinarizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score)
from mlxtend.frequent_patterns import apriori, association_rules

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('All libraries loaded successfully.')

# --- Colab (Python 3.13 image, 2025): jupyter_client spams a
# DeprecationWarning about datetime.utcnow on nearly EVERY output message.
# During grid search this buries progress lines under thousands of warnings
# (the "error.txt" flood). Silence it once, here:
import warnings as _w
_w.filterwarnings('ignore', message='.*utcnow.*')
_w.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nMissing percentages:\n{(df.isnull().sum() / len(df) * 100).round(2)}')
print(f'\nDuplicate rows: {df.duplicated().sum()}')
df.head()

In [ ]:
df.describe()

---
## Phase 2: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['Rating'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Rating Distribution')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Count')

def assign_difficulty(rating):
    if rating <= 1200: return 'Easy'
    elif rating <= 1600: return 'Medium'
    elif rating <= 2000: return 'Hard'
    else: return 'Expert'

df['Difficulty'] = df['Rating'].apply(assign_difficulty)
diff_counts = df['Difficulty'].value_counts()
diff_order = ['Easy', 'Medium', 'Hard', 'Expert']
diff_counts = diff_counts.reindex(diff_order)
diff_counts.plot(kind='bar', ax=axes[0, 1], edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Difficulty Distribution')
axes[0, 1].set_ylabel('Count')
for i, v in enumerate(diff_counts):
    axes[0, 1].text(i, v + 200, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

axes[1, 0].scatter(df['Rating'], df['Popularity'], alpha=0.1, s=5)
axes[1, 0].set_title('Rating vs Popularity')
axes[1, 0].set_xlabel('Rating')
axes[1, 0].set_ylabel('Popularity')

axes[1, 1].scatter(df['Rating'], df['NbPlays'], alpha=0.1, s=5)
axes[1, 1].set_title('Rating vs NbPlays')
axes[1, 1].set_xlabel('Rating')
axes[1, 1].set_ylabel('NbPlays (log)')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()
print(f'\nClass distribution:\n{df["Difficulty"].value_counts().reindex(diff_order)}')

In [ ]:
theme_lists = df['Themes'].fillna('').str.split()
all_themes = [t for themes in theme_lists for t in themes if t]
theme_counts = pd.Series(all_themes).value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
theme_counts.plot(kind='barh', ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_title('Top 20 Themes')
axes[0].set_xlabel('Count')

num_cols = ['Rating', 'RatingDeviation', 'Popularity', 'NbPlays']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
tautological = ['mateIn1', 'mateIn2', 'mateIn3', 'mateIn4', 'mateIn5', 'oneMove', 'short', 'long']
mlb_check = MultiLabelBinarizer()
theme_binary_check = mlb_check.fit_transform(theme_lists.apply(lambda x: [t for t in x if t]))
theme_df_check = pd.DataFrame(theme_binary_check, columns=mlb_check.classes_)

taut_cols = [t for t in tautological if t in theme_df_check.columns]
theme_df_check['Difficulty'] = df['Difficulty']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for i, theme in enumerate(taut_cols):
    ax = axes[i // 4, i % 4]
    theme_df_check.groupby('Difficulty')[theme].mean().reindex(diff_order).plot(kind='bar', ax=ax, alpha=0.7)
    ax.set_title(f'{theme} by Difficulty')
    ax.set_ylabel('Proportion')
plt.tight_layout()
plt.show()

---
## Phase 3: Data Preprocessing

In [ ]:
df_prep = df.drop(columns=['OpeningTags', 'DailyDate', 'PuzzleId', 'FEN', 'Moves', 'GameUrl'])
df_prep['Difficulty'] = df_prep['Rating'].apply(assign_difficulty)
df_prep = df_prep.drop(columns=['Rating'])
print(f'After dropping columns: {df_prep.shape}')
df_prep.head()

In [ ]:
def extract_fen_features(fen):
    try:
        board = chess.Board(fen)
        piece_values = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
                        chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}
        white_material = sum(piece_values[p] for p in board.pieces(chess.WHITE, chess.KNIGHT)) + \
                         sum(piece_values[p] for p in board.pieces(chess.WHITE, chess.BISHOP)) + \
                         sum(piece_values[p] for p in board.pieces(chess.WHITE, chess.ROOK)) + \
                         sum(piece_values[p] for p in board.pieces(chess.WHITE, chess.QUEEN)) + \
                         sum(1 for _ in board.pieces(chess.PAWN, chess.WHITE))
        black_material = sum(piece_values[p] for p in board.pieces(chess.BLACK, chess.KNIGHT)) + \
                         sum(piece_values[p] for p in board.pieces(chess.BLACK, chess.BISHOP)) + \
                         sum(piece_values[p] for p in board.pieces(chess.BLACK, chess.ROOK)) + \
                         sum(piece_values[p] for p in board.pieces(chess.BLACK, chess.QUEEN)) + \
                         sum(1 for _ in board.pieces(chess.PAWN, chess.BLACK))
        white_pawns = len(list(board.pieces(chess.PAWN, chess.WHITE)))
        black_pawns = len(list(board.pieces(chess.PAWN, chess.BLACK)))
        white_king = board.king(chess.WHITE)
        black_king = board.king(chess.BLACK)
        king_distance = 0
        if white_king is not None and black_king is not None:
            king_distance = chess.square_distance(white_king, black_king)
        white_attackers = 0
        black_attackers = 0
        if white_king is not None:
            white_attackers = len(board.attackers(chess.BLACK, white_king))
        if black_king is not None:
            black_attackers = len(board.attackers(chess.WHITE, black_king))
        total_pieces = sum(1 for _ in board.piece_map())
        white_castled = 0
        if white_king is not None:
            white_castled = 1 if chess.square_file(white_king) in [6, 7] else 0
        white_doubled = 0
        for file in range(8):
            pawns = list(board.pieces(chess.PAWN, chess.WHITE))
            if sum(1 for p in pawns if chess.square_file(p) == file) > 1:
                white_doubled += 1
        return pd.Series({
            'WhiteMaterial': white_material, 'BlackMaterial': black_material,
            'MaterialBalance': white_material - black_material, 'TotalPieces': total_pieces,
            'WhitePawns': white_pawns, 'BlackPawns': black_pawns,
            'IsWhiteTurn': 1 if board.turn == chess.WHITE else 0,
            'LegalMoveCount': board.legal_moves.count(), 'InCheck': 1 if board.is_check() else 0,
            'WhiteKingAttackers': black_attackers, 'BlackKingAttackers': white_attackers,
            'KingDistance': king_distance, 'WhiteCastled': white_castled,
            'WhiteDoubledPawns': white_doubled
        })
    except:
        return pd.Series({
            'WhiteMaterial': 0, 'BlackMaterial': 0, 'MaterialBalance': 0,
            'TotalPieces': 0, 'WhitePawns': 0, 'BlackPawns': 0,
            'IsWhiteTurn': 0, 'LegalMoveCount': 0, 'InCheck': 0,
            'WhiteKingAttackers': 0, 'BlackKingAttackers': 0,
            'KingDistance': 0, 'WhiteCastled': 0, 'WhiteDoubledPawns': 0
        })

fen_features = df['FEN'].apply(extract_fen_features)
print(f'FEN features extracted: {fen_features.shape}')
fen_features.head()

In [ ]:
mlb = MultiLabelBinarizer()
theme_binary = mlb.fit_transform(theme_lists.apply(lambda x: [t for t in x if t]))
theme_df = pd.DataFrame(theme_binary, columns=mlb.classes_)
print(f'Theme features: {theme_df.shape}')

numerical_cols = ['RatingDeviation', 'Popularity', 'NbPlays']
X_numerical = df_prep[numerical_cols].copy()

X = pd.concat([X_numerical.reset_index(drop=True),
                fen_features.reset_index(drop=True),
                theme_df.reset_index(drop=True)], axis=1)

y = df_prep['Difficulty'].copy()

print(f'Final feature matrix: {X.shape}')
print(f'Target distribution:\n{y.value_counts().reindex(diff_order)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
num_features = ['RatingDeviation', 'Popularity', 'NbPlays'] + list(fen_features.columns)
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'\nTrain distribution:\n{y_train.value_counts().reindex(diff_order)}')
print(f'\nTest distribution:\n{y_test.value_counts().reindex(diff_order)}')

---
## Phase 4: Association Rule Mining (Descriptive Task)

In [ ]:
theme_assoc = theme_df.copy()
diff_onehot = pd.get_dummies(df['Difficulty'], prefix='Diff').astype(bool)
theme_assoc = pd.concat([theme_assoc, diff_onehot], axis=1)

frequent = apriori(theme_assoc, min_support=0.05, use_colnames=True)
print(f'Frequent itemsets: {len(frequent)}')
frequent.head(10)

In [ ]:
rules = association_rules(frequent, metric='confidence', min_threshold=0.70)
rules = rules[rules['lift'] > 1.0]

diff_cols = {'Diff_Easy', 'Diff_Medium', 'Diff_Hard', 'Diff_Expert'}
rules = rules[rules['consequents'].apply(lambda s: any(c in diff_cols for c in s))]

rules = rules.sort_values('lift', ascending=False)
print(f'Association rules (difficulty in consequent): {len(rules)}')
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(rules['support'], rules['confidence'], alpha=0.6, s=50, c=rules['lift'], cmap='YlOrRd')
axes[0].set_xlabel('Support')
axes[0].set_ylabel('Confidence')
axes[0].set_title('Association Rules: Support vs Confidence')
plt.colorbar(axes[0].collections[0], ax=axes[0], label='Lift')

top10 = rules.head(10)
labels = [f"{list(r['antecedents'])} -> {list(r['consequents'])}" for _, r in top10.iterrows()]
axes[1].barh(range(len(top10)), top10['lift'].values, alpha=0.7)
axes[1].set_yticks(range(len(top10)))
axes[1].set_yticklabels(labels, fontsize=8)
axes[1].set_xlabel('Lift')
axes[1].set_title('Top 10 Rules by Lift')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## Phase 5: Model Training

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
class_names = le.classes_
print(f'Classes: {class_names}')

In [ ]:
import time, os, gc, json, joblib
from sklearn.metrics import f1_score

# ================= Colab survival kit (fixes the crash) =================
# 1) float32 copy of the features: halves RAM and speeds up KNN/SVC/trees.
#    Metrics are unchanged (verified on the 50k sample).
X_train_f32 = X_train.astype(np.float32)
X_test_f32  = X_test.astype(np.float32)

# 2) Checkpoint every finished model to Drive. If Colab dies mid-cell,
#    re-run this cell: finished models are loaded, NOT retrained.
#    (Delete the folder in Drive if you change grids and want a fresh run.)
CKPT_DIR = '/content/drive/MyDrive/Datamining/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
results = {}
_res_path = f'{CKPT_DIR}/cv_results.json'
if os.path.exists(_res_path):
    results = json.load(open(_res_path))

# 3) Parallelize the GRID SEARCH, never the model inside it.
#    Old code: GridSearchCV(n_jobs=2) x RandomForest(n_jobs=2) = 4 threads
#    fighting over 2 vCPUs + duplicate data in each worker process.
N_JOBS = -1

models = {
    'Dummy': DummyClassifier(strategy='most_frequent'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1),
    'KNN': KNeighborsClassifier(),
    'LinearSVC': LinearSVC(max_iter=10000, random_state=42, dual=False)  # dual=False: 32k rows >> 90 features -> fast primal solver
}

param_grids = {
    'Decision Tree': {'max_depth': [10, 15, 20, None], 'min_samples_split': [2, 5, 10]},
    'Random Forest': {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, None]},
    'KNN': {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance']},
    'LinearSVC': {'C': [0.1, 1, 10, 100]}
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_models = {}

def log(msg):
    print(msg, flush=True)   # flush=True: see progress LIVE in Colab output

for name, model in models.items():
    ckpt_path = f'{CKPT_DIR}/{name.replace(" ", "_")}.joblib'
    if os.path.exists(ckpt_path):
        best_models[name] = joblib.load(ckpt_path)
        log(f'\n{name}: loaded from checkpoint (skipped)')
        continue

    log(f'\nTraining {name}...')
    t0 = time.time()
    if name in param_grids:
        grid = GridSearchCV(model, param_grids[name], cv=skf,
                            scoring='f1_macro', n_jobs=N_JOBS, verbose=1)
        grid.fit(X_train_f32, y_train_enc)
        best_models[name] = grid.best_estimator_
        best_idx = grid.best_index_
        results[name] = {
            'cv_f1_mean': float(grid.best_score_),
            'cv_f1_std': float(grid.cv_results_['std_test_score'][best_idx])  # real std (old code hardcoded 0)
        }
        log(f"  Best params: {grid.best_params_}")
        log(f"  Best CV F1 (macro): {results[name]['cv_f1_mean']:.4f} +/- {results[name]['cv_f1_std']:.4f}")
        del grid
    else:
        cv_scores = []
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_f32, y_train_enc), 1):
            model.fit(X_train_f32.iloc[tr_idx], y_train_enc[tr_idx])
            f = f1_score(y_train_enc[va_idx],
                         model.predict(X_train_f32.iloc[va_idx]), average='macro')
            cv_scores.append(f)
            log(f'  fold {fold}/5  F1 (macro) = {f:.4f}')
        best_models[name] = model
        results[name] = {'cv_f1_mean': float(np.mean(cv_scores)),
                         'cv_f1_std': float(np.std(cv_scores))}
        log(f"  CV F1 (macro): {results[name]['cv_f1_mean']:.4f} +/- {results[name]['cv_f1_std']:.4f}")

    log(f'  finished in {time.time() - t0:.1f}s')
    joblib.dump(best_models[name], ckpt_path)                     # survive crashes
    json.dump(results, open(_res_path, 'w'), indent=2, default=float)
    gc.collect()

del X_train_f32   # free the float32 copy; X_test_f32 stays for evaluation cells
gc.collect()


In [ ]:
import time, gc
log('Training RBF SVM on 10k subsample...')
t0 = time.time()

subsample_idx = np.random.RandomState(42).choice(len(X_train), size=10000, replace=False)
X_sub = X_train.iloc[subsample_idx].astype(np.float32)
y_sub = y_train_enc[subsample_idx]

rbf_ckpt = f'{CKPT_DIR}/RBF_SVM.joblib'
if os.path.exists(rbf_ckpt):
    best_models['RBF SVM'] = joblib.load(rbf_ckpt)
    log('  loaded from checkpoint (skipped)')
else:
    # probability=False during the search: SVC(probability=True) runs an
    # internal 5-fold Platt calibration for EVERY grid candidate (~6x slower
    # for no benefit). We calibrate ONCE on the final model instead.
    rbf_svm = SVC(kernel='rbf', probability=False, cache_size=512, random_state=42)
    rbf_grid = GridSearchCV(rbf_svm, {'C': [1, 10, 100], 'gamma': ['scale', 'auto']},
                            cv=3, scoring='f1_macro', n_jobs=N_JOBS, verbose=1)
    rbf_grid.fit(X_sub, y_sub)
    log(f'  Best params: {rbf_grid.best_params_}')
    log(f'  Best CV F1 (macro): {rbf_grid.best_score_:.4f}')

    best_models['RBF SVM'] = SVC(kernel='rbf', probability=True, cache_size=512,
                                 random_state=42, **rbf_grid.best_params_)
    best_models['RBF SVM'].fit(X_sub, y_sub)   # single calibration fit
    results['RBF SVM'] = {'cv_f1_mean': float(rbf_grid.best_score_),
                          'cv_f1_std': float(rbf_grid.cv_results_['std_test_score'][rbf_grid.best_index_])}
    joblib.dump(best_models['RBF SVM'], rbf_ckpt)
    json.dump(results, open(_res_path, 'w'), indent=2, default=float)

log(f'  finished in {time.time() - t0:.1f}s')
gc.collect()


---
## Phase 6: Evaluation and Audit

In [ ]:
eval_results = {}

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test_enc, y_pred)
    prec = precision_score(y_test_enc, y_pred, average='macro')
    rec = recall_score(y_test_enc, y_pred, average='macro')
    f1 = f1_score(y_test_enc, y_pred, average='macro')
    eval_results[name] = {
        'Accuracy': acc, 'Precision (macro)': prec,
        'Recall (macro)': rec, 'F1 (macro)': f1
    }
    print(f'\n{name}:')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1 (macro): {f1:.4f}')

eval_df = pd.DataFrame(eval_results).T
print(f'\n{eval_df}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, (name, model) in enumerate(best_models.items()):
    ax = axes[idx // 3, idx % 3]
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test_enc, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(f'{name}\nF1={eval_results[name]["F1 (macro)"]:.3f}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
if len(best_models) < 6:
    axes[1, 2].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, (name, model) in enumerate(best_models.items()):
    ax = axes[idx // 3, idx % 3]
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)
    elif hasattr(model, 'decision_function'):
        y_prob = model.decision_function(X_test)
        if y_prob.ndim == 1:
            ax.plot([0, 1], [0, 1], 'k--')
            ax.set_title(f'{name}\n(N/A)')
            continue
    else:
        ax.plot([0, 1], [0, 1], 'k--')
        ax.set_title(f'{name}\n(N/A)')
        continue
    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_test_enc == i, y_prob[:, i])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{class_names[i]} (AUC={roc_auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_title(name)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=7)
if len(best_models) < 6:
    axes[1, 2].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
print('Theme Leakage Audit: Run A vs Run B')
print('=' * 50)

tautological_themes = ['mateIn1', 'mateIn2', 'mateIn3', 'mateIn4', 'mateIn5', 'oneMove', 'short', 'long']
non_taut_cols = [c for c in theme_df.columns if c not in tautological_themes]

X_runA = X.copy()
X_runB = X[non_taut_cols + [c for c in X.columns if c not in theme_df.columns]]

X_trA, X_teA, y_trA, y_teA = train_test_split(X_runA, y, test_size=0.2, random_state=42, stratify=y)
X_trB, X_teB, y_trB, y_teB = train_test_split(X_runB, y, test_size=0.2, random_state=42, stratify=y)

y_trA_enc = le.transform(y_trA)
y_teA_enc = le.transform(y_teA)
y_trB_enc = le.transform(y_trB)
y_teB_enc = le.transform(y_teB)

scalerA = StandardScaler()
scalerB = StandardScaler()

num_f_A = [c for c in num_features if c in X_trA.columns]
X_trA[num_f_A] = scalerA.fit_transform(X_trA[num_f_A])
X_teA[num_f_A] = scalerA.transform(X_teA[num_f_A])

num_f_B = [c for c in num_features if c in X_trB.columns]
X_trB[num_f_B] = scalerB.fit_transform(X_trB[num_f_B])
X_teB[num_f_B] = scalerB.transform(X_teB[num_f_B])

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=2)
rf.fit(X_trA, y_trA_enc)
accA = accuracy_score(y_teA_enc, rf.predict(X_teA))
f1A = f1_score(y_teA_enc, rf.predict(X_teA), average='macro')

rf.fit(X_trB, y_trB_enc)
accB = accuracy_score(y_teB_enc, rf.predict(X_teB))
f1B = f1_score(y_teB_enc, rf.predict(X_teB), average='macro')

print(f'\nRun A (all themes):     Accuracy={accA:.4f}, F1={f1A:.4f}')
print(f'Run B (no tautological): Accuracy={accB:.4f}, F1={f1B:.4f}')
print(f'\nAccuracy drop: {(accA - accB)*100:.2f} percentage points')
print(f'F1 drop:       {(f1A - f1B)*100:.2f} percentage points')

if (accA - accB) * 100 < 3:
    print('Interpretation: Model learned genuine positional difficulty.')
elif (accA - accB) * 100 < 8:
    print('Interpretation: Some shortcut learning detected.')
else:
    print('Interpretation: Heavy shortcut learning. Model relies on tautological themes.')

In [ ]:
rf_full = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_full.fit(X_trA, y_trA_enc)

importances = pd.Series(rf_full.feature_importances_, index=X_trA.columns)
top15 = importances.nlargest(15)

plt.figure(figsize=(10, 8))
top15.sort_values().plot(kind='barh', alpha=0.7)
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
y_pred_final = rf_full.predict(X_teA)
misclassified_idx = np.where(y_pred_final != y_teA_enc)[0]

if len(misclassified_idx) > 20:
    sample_idx = np.random.RandomState(42).choice(misclassified_idx, size=20, replace=False)
else:
    sample_idx = misclassified_idx

print(f'Total misclassified: {len(misclassified_idx)} / {len(y_teA_enc)}')
print(f'\nSample of 20 misclassified examples:')
print('=' * 80)

original_indices = X_teA.index[sample_idx]
for i, idx in enumerate(original_indices):
    row = df.loc[idx]
    pred_label = class_names[y_pred_final[sample_idx[i]]]
    true_label = y_teA.iloc[sample_idx[i]]
    print(f'{i+1:2d}. PuzzleId: {row["PuzzleId"]}')
    print(f'    Rating: {row["Rating"]}, True: {true_label}, Predicted: {pred_label}')
    print(f'    Themes: {row["Themes"]}')
    print(f'    FEN: {row["FEN"][:60]}...')
    print()

---
## Summary

In [ ]:
print('MODEL COMPARISON SUMMARY')
print('=' * 60)
print(eval_df.to_string())
print()
best_model_name = eval_df['F1 (macro)'].idxmax()
print(f'Best model: {best_model_name} (F1 macro = {eval_df.loc[best_model_name, "F1 (macro)"]:.4f})')
print(f'\nLeakage audit drop: {(accA - accB)*100:.2f} pp')
print(f'Top 5 features: {list(top15.index[:5])}')